In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
base_path = "s3://delivery-data-mohit/"
files = dbutils.fs.ls(base_path)

for file in files:
    
    if file.name.endswith(".csv"):
        
        file_path = file.path
        table_name = file.name.replace(".csv", "")
        
        target_table = f"delivery_cata.bronze.{table_name}"
        
        # Read schema from delivery_ddl
        ddl_table = f"delivery_cata.delivery_ddl.{table_name}"
        target_schema = spark.table(ddl_table).schema
        
        # Read CSV from S3
        df = spark.read \
            .option("inferschema",'false') \
            .schema(target_schema) \
            .option("header", "true") \
            .csv(file_path)
        
        # Add metadata
        df = df.withColumn("_ingested_at", current_timestamp()) \
               .withColumn("_source_file", lit(file.name))
        
        # Write to Delta table
        df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(target_table)
        
        print(f"Loded table: {table_name}")

Loded table: areas_raw
Loded table: deliveries_raw
Loded table: drivers_raw
Loded table: orders_raw
Loded table: payments_raw
Loded table: users_raw
Loded table: vehicles_raw
